In [ ]:
from validate import validation_metrics
import os
import pickle
from ttt_optimized import *

plt.style.use('utils\plotstyle.mplstyle')#

c:\Users\leonardo\anaconda3\envs\doutorado\lib\site-packages\albumentations\__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.8' (you have '2.0.5'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [2]:
path_icopevid = 'Datasets\\Originais\\iCOPE\\iCOPEvid'
path_icopevid_frames = 'Datasets\\Originais\\iCOPE\\iCOPEvid\\all_frames'

In [5]:
model_name = "ViT_B_32_ENSEMBLE_FINAL"
path_experiments = 'experiments\\' + model_name

with open(os.path.join(path_experiments,'icopevid',f'results_ensemble_10.pkl'), 'rb') as f:
    results_video = pickle.load(f)


mcdp = True

In [ ]:
xai_root = Path(path_experiments) / "icopevid" / "XAI"

arrays = run_pain_sign_report(
     results_video=results_video,
     model_name=model_name,
     path_icopevid_frames=path_icopevid_frames,
     xai_root=xai_root,
     out_dir=r"C:\Users\leonardo\Desktop\icopevid_results",
     mcdp=mcdp,
     ma_window=30,
     duration_s=20.0,
     frame_step=30,
     region_frame_step=1,
     include_region_curves=True,
     region_top_k=-1,
     region_selection="mean",
     region_smooth_window=30,
     xai_alpha=0.6,
     include_mesh_regions=True
)

print(validation_metrics(arrays["preds"], arrays["probs"], arrays["labels"]))







Frames S001_Rest_1_[0]_20s: 100%|██████████| 600/600 [01:03<00:00,  9.47it/s]


In [ ]:

corr_df, summary_df = compute_pain_region_correlations(
    results_video=results_video,
    model_name=model_name,
    path_icopevid_frames=path_icopevid_frames,
    xai_root=xai_root,
    mcdp=mcdp,
    duration_s=20.0,
    ma_window=30,
    region_frame_step=1,
    region_smooth_window=30,
    method="pearson",
)

print("\nPer-video correlations (pain sign vs XAI regions):")
print(corr_df.round(3).to_string())

summary_fmt = summary_df.copy()
summary_fmt["mean_std"] = summary_fmt.apply(
    lambda r: f"{r['mean']:.3f} +/- {r['std']:.3f} (n={int(r['n'])})" if pd.notna(r["mean"]) else "nan",
    axis=1,
)
print("\nMean +/- Std per region:")
print(summary_fmt["mean_std"].to_string())

In [ ]:
video_name = list(results_video.keys())[10]
out_path = render_pain_sign_animation(
    video_name=video_name,
    video_results=results_video[video_name],
    model_name=model_name,
    video_dir=Path(path_icopevid_frames) / video_name,
    xai_root=Path(path_experiments) / "icopevid" / "XAI",
    out_path=r"C:\Users\leonardo\Desktop\icopevid_results\pain_sign_anim.mp4",
    mcdp=mcdp,
    fps=30,                # set to your real FPS if known
    duration_s=20.0,       # keep consistent with your plots
    frame_step=1,          # >1 will skip frames but keep real-time speed
    include_region_curves=True,
    region_top_k=-1,
)
print(out_path)
